# NB-B1: CardioFormer — Domain-Specific Cardiac Transformer


## Why CardioFormer?
RNNs (NB-A2) are excellent for temporal sequences, but Transformers allow for **global context** within a cardiac sequence. A Transformer can attend to a subtle P-wave abnormality 4 beats ago to inform the current classification, regardless of distance.

## Objectives
1. **Clinical Tokenization**: Project each heartbeat into a high-dimensional "token" space.
2. **Self-Attention Mechanism**: Compute global dependencies across the 8-beat sequence.
3. **Adaptive Positional Encoding**: Help the model understand the temporal order of beats.
4. **Inter-Patient Validation**: Train {100-108}, Test {200, 201}.

Multi-modal self-attention feature weighting and clinical tokenization for transformer-based physiological signal analysis.

### Setting the Stage: Transformers for the Heart
This notebook is quite special—I'm building 'CardioFormer'. Standard CNNs are great for local patterns, but Transformers excel at global context. I'm importing PyTorch and checking my environment as usual. The aim is to treat heartbeats as 'tokens' in a sentence, where the rhythm is the story.

In [1]:
# Cell 1: Dependency Verification
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import classification_report, f1_score
import wfdb, math, os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch: {torch.__version__} | Device: {device}")


PyTorch: 2.5.1+cu121 | Device: cuda


###  Tokenizing the Rhythms: Data Pipeline
I'm preparing sequences of 8 beats again. In the Transformer world, these are my tokens. I'm carefully loading the training and test sets from separate patients to ensure the model really learns generalizable cardiac features rather than just memorizing specific heart patterns.

In [2]:
# Cell 2: Data Loading (8-Beat Temporal Sequences)
WINDOW_SIZE   = 300
SEQ_LEN       = 8
BATCH_SIZE    = 32
TRAIN_RECORDS = ['100', '101', '102', '103', '106', '108']
TEST_RECORDS  = ['200', '201']
LABEL_MAP     = {'N': 0, 'L': 1, 'R': 2, 'V': 3}
CLASS_NAMES   = ['Normal (N)', 'LBBB (L)', 'RBBB (R)', 'PVC (V)']

def load_beat_sequences(record_ids, window_size=300, seq_len=8):
    X_all, y_all = [], []
    for rid in record_ids:
        try:
            record = wfdb.rdrecord(rid, pn_dir='mitdb')
            ann    = wfdb.rdann(rid, extension='atr', pn_dir='mitdb')
            signal = record.p_signal[:, 0]
            beats, labels = [], []
            for i, peak in enumerate(ann.sample):
                if (ann.symbol[i] in LABEL_MAP and peak > window_size//2 and peak < len(signal)-window_size//2):
                    beats.append(signal[peak-window_size//2 : peak+window_size//2])
                    labels.append(LABEL_MAP[ann.symbol[i]])
            
            if len(beats) >= seq_len:
                for i in range(len(beats) - seq_len):
                    X_all.append(beats[i : i + seq_len])
                    y_all.append(labels[i + seq_len - 1])
                print(f"  Record {rid}: {len(beats)} beats -> {len(beats)-seq_len} sequences")
        except Exception as e:
            print(f"  [WARN] {rid}: {e}")
    return np.array(X_all, dtype=np.float32), np.array(y_all)

print("Loading Training Data...")
X_train, y_train = load_beat_sequences(TRAIN_RECORDS)
print("\nLoading Test Data...")
X_test, y_test   = load_beat_sequences(TEST_RECORDS)

train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train)), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test)), batch_size=BATCH_SIZE)
print(f"\nBatches: {len(train_loader)} training, {len(test_loader)} testing")


Loading Training Data...
  Record 100: 2238 beats -> 2230 sequences
  Record 101: 1859 beats -> 1851 sequences
  Record 102: 103 beats -> 95 sequences
  Record 103: 2081 beats -> 2073 sequences
  Record 106: 2027 beats -> 2019 sequences
  Record 108: 1755 beats -> 1747 sequences

Loading Test Data...
  Record 200: 2568 beats -> 2560 sequences
  Record 201: 1823 beats -> 1815 sequences

Batches: 313 training, 137 testing


###  The Attention Engine: CardioFormer Architecture
This is the core logic. I've implemented a Positional Encoding layer because Transformers, unlike GRUs, don't naturally know the order of tokens. The 'CardioFormer' uses Multi-Head Attention to look at different parts of the 8-beat sequence simultaneously. It's like having multiple doctors focus on different aspects of the same ECG recording at once.

In [3]:
# Cell 3: CardioFormer Architecture
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
    def forward(self, x):
        return x + self.pe[:x.size(1), :]

class CardioFormer(nn.Module):
    def __init__(self, d_model=128, nhead=8, num_layers=4, num_classes=4):
        super().__init__()
        # Beat Tokenizer (Projects 300 samples -> 128 embedding)
        self.tokenizer = nn.Sequential(
            nn.Conv1d(1, 16, 7, padding=3), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(16, 32, 5, padding=2), nn.ReLU(), nn.MaxPool1d(4), nn.Flatten(),
            nn.Linear(32 * (WINDOW_SIZE // 16), d_model)
        )
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=512, dropout=0.2, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, num_classes)
        )
    def forward(self, x):
        # x: (batch, 8, 300)
        b, s, w = x.shape
        tokens = self.tokenizer(x.view(b*s, 1, w)).view(b, s, -1)
        tokens = self.pos_encoder(tokens)
        trans_out = self.transformer(tokens)
        # Global Average Pooling across the 8 beats
        pooled = trans_out.mean(dim=1)
        return self.classifier(pooled)

model_b1 = CardioFormer().to(device)
print(f"CardioFormer initialized | Parameters: {sum(p.numel() for p in model_b1.parameters()):,}")


CardioFormer initialized | Parameters: 878,180


###  Training the CardioFormer
Time to train! I'm using class weights once more to handle the imbalance in MIT-BIH recordings. Watch as the Transformer learns to attend to the right beats in the sequence. I'm tracking the progress epoch by epoch, aiming for a model that can handle complex multi-beat dependencies.

In [4]:
# Cell 4: Training Loop
class_weights = torch.tensor([0.5, 1.5, 1.5, 4.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model_b1.parameters(), lr=1e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=5e-4, steps_per_epoch=len(train_loader), epochs=10)

print("Starting CardioFormer training...")
for epoch in range(1, 11):
    model_b1.train()
    tl, c, tot = 0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        out = model_b1(bx)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step(); scheduler.step()
        tl += loss.item(); c += (out.argmax(1)==by).sum().item(); tot += by.size(0)
    print(f"Epoch {epoch:02d} | Loss: {tl/len(train_loader):.4f} | Acc: {100*c/tot:.1f}%")


Starting CardioFormer training...
Epoch 01 | Loss: 0.7732 | Acc: 88.8%
Epoch 02 | Loss: 0.2127 | Acc: 95.6%
Epoch 03 | Loss: 0.1004 | Acc: 99.0%
Epoch 04 | Loss: 0.0623 | Acc: 99.4%
Epoch 05 | Loss: 0.0470 | Acc: 99.8%
Epoch 06 | Loss: 0.0570 | Acc: 99.5%
Epoch 07 | Loss: 0.0427 | Acc: 99.8%
Epoch 08 | Loss: 0.0353 | Acc: 99.8%
Epoch 09 | Loss: 0.0345 | Acc: 99.9%
Epoch 10 | Loss: 0.0246 | Acc: 99.9%


### Clinical Performance & Weights Persistence
Last step: evaluation on the test patients. I'm checking if the Transformer's broader perspective pays off in accuracy. Once satisfied, I'm saving the model weights so Pulse-Mind can leverage this Transformer-based intelligence in its ensemble later.

In [5]:
# Cell 5: Inter-Patient Evaluation
model_b1.eval()
preds, labs = [], []
with torch.no_grad():
    for bx, by in test_loader:
        preds.extend(model_b1(bx.to(device)).argmax(1).cpu().numpy())
        labs.extend(by.numpy())
preds, labs = np.array(preds), np.array(labs)

pcls = sorted(np.unique(np.concatenate([labs, preds])))
pnames = [CLASS_NAMES[i] for i in pcls if i < len(CLASS_NAMES)]
print("="*60)
print(f"Accuracy: {(preds==labs).mean()*100:.2f}%")
print(classification_report(labs, preds, labels=pcls, target_names=pnames, zero_division=0))

torch.save(model_b1.state_dict(), '../ai_training/output/NB_B1_CardioFormer.pth')
print("CardioFormer weights saved.")


Accuracy: 95.57%
              precision    recall  f1-score   support

  Normal (N)       0.95      1.00      0.97      3355
     PVC (V)       0.98      0.82      0.90      1020

    accuracy                           0.96      4375
   macro avg       0.97      0.91      0.93      4375
weighted avg       0.96      0.96      0.95      4375

CardioFormer weights saved.
